<div dir="rtl">

<h1>«معاكِ»: تجهيز صور BCMID من Zenodo</h1>

<p><b>الدفتر ده بيتشغل مرة واحدة، قبل الدفتر الأساسي <code>maaki_models_reviewed.ipynb</code>.</b></p>

<p><b>ليه؟</b> مساحة جوجل المجانية 15 GB، والداتا لوحدها حوالي 13 GB، فمش كل الصور اتحطت على الدرايف. والدفتر الأساسي مش محتاج من الصورة الخام غير حاجة واحدة: الميزات اللي ResNet50 بتطلّعها منها. فبدل ما نخزن الصور، بنحسب الميزات ونخزنها، وحجمها كام ميجا بس.</p>

<p><b>بيعمل إيه:</b></p>
<ol>
<li>بينزّل الداتا من مصدرها الرسمي (Zenodo، النسخة 3، DOI 10.5281/zenodo.19402690، رخصة CC BY 4.0) على هارد Colab المؤقت، مش على الدرايف.</li>
<li>بيعمل الفهرس الرسمي لكل صور الداتا من الملف نفسه، ويحفظه في <code>maaki_outputs/bcmid_manifest.csv</code>.</li>
<li>بيدوّر على الصور اللي مالهاش ميزات محفوظة، ويفكها على هارد Colab بس.</li>
<li>بيجهزها ويطلّع ميزاتها بنفس دوال التجهيز ونفس ResNet50 اللي في الدفتر الأساسي بالحرف، ويضيفها لملفات الميزات المحفوظة.</li>
</ol>

<p><b>الوقت:</b> التنزيل خد حوالي ساعة ونص في تجربتنا (حسب سرعة Zenodo)، وحساب ميزات الـ 358 صورة ماموجرام الناقصة بياخد حوالي 20 لـ 30 دقيقة على CPU. ماتغيريش الـ Runtime بين الخليتين، لأن الملف اللي نزل بيتمسح مع تغيير الـ Runtime. ولو مفيش ميزات محفوظة خالص، اختاري GPU علشان الحساب يبقى أسرع.</p>

<p><b>اتأكدنا إن النتيجة واحدة:</b> على داتا تجريبية، النوت بوك الأساسي طلّع نفس الأرقام بالظبط في الحالتين: لما كل الصور كانت على الدرايف، ولما جزء منها اتحسب من الدفتر ده.</p>

</div>

In [ ]:
# 1) تنزيل BCMID من مصدرها الرسمي على هارد Colab (مش على الدرايف)
from google.colab import drive
drive.mount('/content/drive')
URL = 'https://zenodo.org/records/19402690/files/BCMID.zip?download=1'   # BCMID v3, CC BY 4.0
ZIP = '/content/BCMID.zip'

!wget -c -q --show-progress --progress=bar:force:noscroll --tries=20 --read-timeout=60 --retry-on-http-error=429,500,502,503,504 -O {ZIP} "{URL}"

<div dir="rtl"><h3>2) ميزات الصور الناقصة</h3><p>في الآخر المفروض كل صورة ليها ميزات: الماموجرام 951 من 951، والسونار 1049 من 1049 (فيه صورة سونار فاضية في الداتا نفسها)، والماموجرام بالمربعات 951 من 951.</p></div>

In [ ]:
# نحسب ميزات الصور اللي مالهاش ميزات محفوظة من الملف اللي نزل، ونحفظ في الدرايف الميزات بس (كام MB).
# التجهيز وResNet50 هنا نفس اللي في النوت بوك بالظبط (ب-2 وب-3 وب-4-ب).
import os, re, shutil, zipfile
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from google.colab import drive
from tqdm.auto import tqdm

drive.mount('/content/drive')
ZIP = '/content/BCMID.zip'
DRIVE = '/content/drive/MyDrive'
BCMID_DIR = os.path.join(DRIVE, 'BCMID')         # اسم الصورة في الميزات = مكانها في الدرايف، زي النوت بوك
OUT_DIR = os.path.join(DRIVE, 'maaki_outputs')
LOCAL = '/content/bcmid_local'                     # الصور الناقصة بتتفك هنا، على هارد Colab مش الدرايف

# 1) القائمة الرسمية لصور الداتا، من ملف Zenodo نفسه
z = zipfile.ZipFile(ZIP)
pat = re.compile(r'(?:^|/)([^/]+)/(Mammogram|Ultrasound)/([^/]+)$')
rows = []
for info in z.infolist():
    name = info.filename.replace('\\', '/')
    m = pat.search(name)
    if info.is_dir() or not m or '__MACOSX' in name:
        continue
    case, modality, file = m.groups()
    if not file.startswith('.') and file.lower().endswith(('.png', '.jpg', '.jpeg')):
        rows.append({'case': case, 'modality': modality, 'file': file, 'bytes': info.file_size, 'member': info.filename})
manifest = pd.DataFrame(rows).sort_values(['case', 'modality', 'file']).reset_index(drop=True)
manifest['path'] = [os.path.join(BCMID_DIR, c, m, f) for c, m, f in zip(manifest['case'], manifest['modality'], manifest['file'])]
manifest[['case', 'modality', 'file', 'bytes']].to_csv(os.path.join(OUT_DIR, 'bcmid_manifest.csv'), index=False)
print('قائمة الصور:', manifest.groupby('modality').size().to_dict())

# 2) الميزات المحفوظة من المرات اللي فاتت، واللي ناقص منها
CACHE = os.path.join(OUT_DIR, 'bcmid_resnet50_features_v2.pkl')
TILE_CACHE = os.path.join(OUT_DIR, 'bcmid_mammo_tiles_v1.pkl')
feat = pd.read_pickle(CACHE) if os.path.exists(CACHE) else pd.DataFrame()
tile_feat = pd.read_pickle(TILE_CACHE) if os.path.exists(TILE_CACHE) else pd.DataFrame()
usable = manifest[manifest['bytes'] > 0].copy()                       # من غير الصورة الفاضية
usable['need_whole'] = ~usable['path'].isin(feat.index)
usable['need_tiles'] = (usable['modality'] == 'Mammogram') & ~usable['path'].isin(tile_feat.index)
todo = usable[usable['need_whole'] | usable['need_tiles']]
print(f"محتاجين ميزات لـ {len(todo)} صورة: {int(todo['need_whole'].sum())} بالصورة كلها، "
      f"و{int(todo['need_tiles'].sum())} بالمربعات")

# 3) نفك الصور دي بس على هارد Colab
for r in tqdm(list(todo.itertuples()), desc='بنفك الصور'):
    target = os.path.join(LOCAL, r.case, r.modality, r.file)
    if not (os.path.exists(target) and os.path.getsize(target) == r.bytes):
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(r.member) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst, 16 * 1024 * 1024)

# 4) دوال التجهيز: منسوخة من النوت بوك بالحرف
IMG_SIZE = 224


def to_gray_uint8(path):
    im = Image.open(path)
    if im.mode in ('I;16', 'I;16B', 'I;16L', 'I', 'F'):          # صور 16-bit
        a = np.asarray(im, dtype=np.float32)
        lo, hi = np.percentile(a, (0.5, 99.5))
        return (np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
    return np.asarray(im.convert('L'))


def square_resize(a, size=IMG_SIZE, align_left=False):
    h, w = a.shape
    s = max(h, w)
    canvas = np.zeros((s, s), dtype=np.uint8)
    top = (s - h) // 2
    left = 0 if align_left else (s - w) // 2
    canvas[top:top + h, left:left + w] = a
    return np.asarray(Image.fromarray(canvas).resize((size, size), Image.Resampling.BILINEAR))


def largest_region_box(a, thr, step, opening, ignore_edges):
    # أكبر منطقة فاتحة في الصورة (على نسخة مصغرة علشان السرعة)
    mask = a[::step, ::step] > thr
    if ignore_edges:                                  # خطوط الإطار على حواف الماموجرام
        mh, mw = max(1, mask.shape[0] // 50), max(1, mask.shape[1] // 50)
        mask[:mh] = False
        mask[-mh:] = False
        mask[:, :mw] = False
        mask[:, -mw:] = False
    mask = ndimage.binary_opening(mask, iterations=opening)
    labels, n = ndimage.label(mask)
    if n == 0:
        return None
    sizes = ndimage.sum(mask, labels, range(1, n + 1))
    ys, xs = np.where(labels == int(np.argmax(sizes)) + 1)
    return ys.min() * step, (ys.max() + 1) * step, xs.min() * step, (xs.max() + 1) * step


def prep_mammogram(path):
    a = to_gray_uint8(path)
    k = max(1, a.shape[1] // 20)
    left, right = a[:, :k], a[:, -k:]
    bg_is_left = left.std() < right.std()           # الخلفية = الجنب الهادي اللي مفيهوش نسيج
    bg_level = float(np.median(left if bg_is_left else right))
    if bg_level > 127:                               # خلفية بيضا = الصورة معكوسة
        a = 255 - a
        bg_level = 255 - bg_level
    if bg_is_left:                                   # جدار الصدر يبقى على الشمال في كل الصور
        a = a[:, ::-1]
    box = largest_region_box(a, thr=bg_level + 25, step=8, opening=2, ignore_edges=True)
    if box is not None:
        r0, r1, c0, c1 = box
        a = a[r0:r1, c0:c1]
    return square_resize(a, align_left=True)


def prep_ultrasound(path):
    a = to_gray_uint8(path)
    box = largest_region_box(a, thr=12, step=2, opening=3, ignore_edges=False)
    if box is not None:
        r0, r1, c0, c1 = box
        if (r1 - r0) * (c1 - c0) > 0.2 * a.size:      # لو المنطقة صغيرة قوي، سيبي الصورة زي ما هي
            a = a[r0:r1, c0:c1]
    return square_resize(a)


GRID = 3
BIG = IMG_SIZE * GRID                     # 672 × 672 بدل 224 × 224


def prep_mammogram_big(path):
    # نفس تجهيز الماموجرام بالظبط، بس بمقاس أكبر
    a = to_gray_uint8(path)
    k = max(1, a.shape[1] // 20)
    left, right = a[:, :k], a[:, -k:]
    bg_is_left = left.std() < right.std()
    bg_level = float(np.median(left if bg_is_left else right))
    if bg_level > 127:
        a = 255 - a
        bg_level = 255 - bg_level
    if bg_is_left:
        a = a[:, ::-1]
    box = largest_region_box(a, thr=bg_level + 25, step=8, opening=2, ignore_edges=True)
    if box is not None:
        r0, r1, c0, c1 = box
        a = a[r0:r1, c0:c1]
    return square_resize(a, size=BIG, align_left=True)


# الصورة الكبيرة بتتقري مرة واحدة بس بدل مرتين (للصورة كلها وللمربعات)، والنتيجة هي هي
_read_image = to_gray_uint8
_last = {}


def to_gray_uint8(path):
    if path not in _last:
        _last.clear()
        _last[path] = _read_image(path)
    return _last[path].copy()


# 5) ResNet50 زي النوت بوك
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

cnn = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(IMG_SIZE, IMG_SIZE, 3))
PREP = {'Mammogram': prep_mammogram, 'Ultrasound': prep_ultrasound}


def run_batch(batch):
    x = preprocess_input(np.stack(batch).astype('float32'))
    return cnn.predict(x, verbose=0)


def tile_feature(path):
    g = prep_mammogram_big(path)
    tiles = [g[r:r + IMG_SIZE, c:c + IMG_SIZE] for r in range(0, BIG, IMG_SIZE) for c in range(0, BIG, IMG_SIZE)]
    tiles = [t for t in tiles if (t > 40).mean() > 0.2] or tiles        # المربعات اللي فيها نسيج بس
    x = preprocess_input(np.stack([np.repeat(t[..., None], 3, axis=-1) for t in tiles]).astype('float32'))
    return cnn.predict(x, verbose=0).max(axis=0)                         # أقوى إشارة في أي مربع


# 6) نحسب الميزات (على CPU بياخد حوالي 20 دقيقة)
batch, keys, parts, tile_rows = [], [], [], {}
for r in tqdm(list(todo.itertuples()), desc='بنحسب الميزات'):
    src = os.path.join(LOCAL, r.case, r.modality, r.file)
    try:
        if r.need_whole:
            g = PREP[r.modality](src)
            batch.append(np.repeat(g[..., None], 3, axis=-1))
            keys.append(r.path)
            if len(batch) == 16:
                parts.append(pd.DataFrame(run_batch(batch), index=keys))
                batch, keys = [], []
        if r.need_tiles:
            tile_rows[r.path] = tile_feature(src)
    except Exception as e:
        print(f'⚠️ تخطّينا {r.file}: {e}')
if batch:
    parts.append(pd.DataFrame(run_batch(batch), index=keys))

# 7) نحفظ الميزات في الدرايف
if parts:
    feat = pd.concat([feat] + parts)
    feat.to_pickle(CACHE)
if tile_rows:
    tile_feat = pd.concat([tile_feat, pd.DataFrame.from_dict(tile_rows, orient='index')])
    tile_feat.to_pickle(TILE_CACHE)

# 8) المراجعة: المفروض كل صورة ليها ميزات
for modality in ('Mammogram', 'Ultrasound'):
    part = usable[usable['modality'] == modality]
    print(f"{modality}: ليها ميزات {int(part['path'].isin(feat.index).sum())} من {len(part)}")
mammo = usable[usable['modality'] == 'Mammogram']
print(f"Mammogram بالمربعات: ليها ميزات {int(mammo['path'].isin(tile_feat.index).sum())} من {len(mammo)}")
print('بنستنى لحد ما الميزات تتحفظ في الدرايف...')
drive.flush_and_unmount()
print('✅ خلصنا')